## Requirements

- Activation Unit
    1st hidden layer: Relu
    2nd hidden layer: Relu
    Output layer: Sigmoid
    
- Learning Rate: 0.001
    
- Momentum: 0.0

- Number of Iterations: 1500

- Batch Size: 100

- Number of hidden layers: 2

- Nodes in Each layer
    Input Layer: 784
    1st hidden layer: 512
    2nd hidden layer: 512
    Output layer: 10
    
Dataset: MNIST 

## 1. Importing Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import math

from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical

## 2. Importing Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [ ]:
# import gzip
# import sys
# import pickle

# f = gzip.open('mnist.pkl.gz', 'rb')
# if sys.version_info < (3,):
#     data = pickle.load(f)
# else:
#     data = pickle.load(f, encoding='bytes')
# f.close()

# (train_images, train_labels), (test_images, test_labels) = data

## 3. Data Exploration

In [ ]:
print("Train Data Shape:", train_images.shape)
print("Train Data Shape:", train_labels.shape)
print('\n')
print("Test Data Shape:", test_images.shape)
print("Test Data Shape:", test_labels.shape)

In [ ]:
print("Number of Classes: ", len(np.unique(train_labels)))

In [ ]:
# display the first image in the training data
plt.imshow(train_images[0,:,:], cmap='gray')
plt.title('Ground Truth : {}'.format(train_labels[0]))
plt.show()

## Data Preparation

In [ ]:
#process the data
#1. convert each image of shape 28*28 to 784 dimensional which will be fed to the network as a single vector
dimData = np.prod(train_images.shape[1:])
print("Number of input features: ", dimData)

In [ ]:
train_data = train_images.reshape(train_images.shape[0],dimData)
test_data = test_images.reshape(test_images.shape[0],dimData)

In [ ]:
print("Train Data Shape:", train_data.shape)
print("Train Data Shape:", train_labels.shape)
print('\n')
print("Test Data Shape:", test_data.shape)
print("Test Data Shape:", test_labels.shape)

In [ ]:
# convert data to float and scale values between 0 and 1
train_data = train_data.astype('float')
test_data = test_data.astype('float')

In [ ]:
# scale data
train_data /=255.0
test_data /=255.0

In [ ]:
# Making sure the data looks good after scaling
plt.imshow(train_data[0].reshape(28, 28), cmap='gray')
plt.title('Ground Truth : {}'.format(train_labels[0]))
plt.show()

In [ ]:
#change the labels frominteger to one-hot encoding
train_labels_one_hot = to_categorical(train_labels)
test_labels_one_hot = to_categorical(test_labels)

## Model Building

In [ ]:
from tensorflow.python.framework import ops

In [ ]:
def create_placeholders(n_X, n_Y):
    X = tf.placeholder(tf.float32, shape = (None, n_X))
    Y = tf.placeholder(tf.float32, shape = (None, n_Y))
    return X, Y

In [ ]:
def initialize_parameters(layers_dims):
    L = len(layers_dims)
    parameters = {}
    for l in range(1, L):
        parameters['W'+str(l)] = tf.get_variable(name = 'W'+str(l), shape = [layers_dims[l-1], layers_dims[l]], initializer = tf.contrib.layers.xavier_initializer(seed = 1))
        parameters['b'+str(l)] = tf.get_variable('b'+str(l), shape = [1, layers_dims[l]], initializer = tf.zeros_initializer())
    return parameters

In [ ]:
def forward_propagation(X, parameters):
    
    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']
    W3 = parameters['W3']
    b3 = parameters['b3']
        
    Z1 = tf.add(tf.matmul(X, W1), b1)
    A1 = tf.nn.relu(Z1)
    Z2 = tf.add(tf.matmul(A1, W2), b2)
    A2 = tf.nn.relu(Z2)
    Z3 = tf.add(tf.matmul(A2, W3), b3)
    
    return A2, Z3

In [ ]:
def compute_cost(Z3, Y):
    cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits_v2(labels = Y, logits = Z3))
    return cost

In [ ]:
def random_mini_batches(X, Y, mini_batch_size = 100, seed = 0):
    
    m = X.shape[0]      
    mini_batches = []
    np.random.seed(seed)
    
    permutation = list(np.random.permutation(m))
    shuffled_X = X[permutation, :]
    shuffled_Y = Y[permutation, :].reshape((m, Y.shape[1]))

    num_complete_minibatches = math.floor(m/mini_batch_size)
    
    for k in range(0, num_complete_minibatches):
        mini_batch_X = shuffled_X[k * mini_batch_size : k * mini_batch_size + mini_batch_size, :]
        mini_batch_Y = shuffled_Y[k * mini_batch_size : k * mini_batch_size + mini_batch_size, :]
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    if m % mini_batch_size != 0:
        mini_batch_X = shuffled_X[num_complete_minibatches * mini_batch_size : m, :]
        mini_batch_Y = shuffled_Y[num_complete_minibatches * mini_batch_size : m, :]
        mini_batch = (mini_batch_X, mini_batch_Y)
        mini_batches.append(mini_batch)
    
    return mini_batches

In [ ]:
def model(X_train, Y_train, X_test, Y_test, num_features, num_classes, layers_dims, learning_rate = 0.001, momentum = 0.0, num_epochs = 1500, minibatch_size = 100, print_cost = True):
    
    ops.reset_default_graph()
    
    tf.set_random_seed(1)
    seed = 3
    
    m = X_train.shape[0]
    n_X = num_features
    n_Y = num_classes
    
    train_costs = []
    test_costs = []
    train_acc = []
    test_acc = []
    #sample_parameters_list = {}
    #cache = np.ones(shape=(n_X,))
    
    X, Y = create_placeholders(n_X, n_Y)
    parameters = initialize_parameters(layers_dims)
    
    A2, Z3 = forward_propagation(X, parameters)
    cost = compute_cost(Z3, Y)
    optimizer = tf.contrib.optimizer_v2.RMSPropOptimizer(learning_rate = learning_rate, momentum = momentum).minimize(cost)
    
    init = tf.global_variables_initializer()
    
    predictions = tf.equal(tf.argmax(Z3, 1), tf.argmax(Y, 1))
    accuracy = tf.reduce_mean(tf.cast(predictions, "float"))
    
    comp_cost = tf.reduce_mean(compute_cost(Z3, Y))
    
    with tf.Session() as sess:
        sess.run(init)
        for epoch in range(num_epochs):
            epoch_cost = 0
            num_minibatches = int(m/minibatch_size)
            seed = seed + 1
            minibatches = random_mini_batches(X_train, Y_train, minibatch_size, seed)
            
            for minibatch in minibatches:
                
                (minibatch_X, minibatch_Y) = minibatch
                _, minibatch_cost = sess.run([optimizer, cost], feed_dict={X:minibatch_X, Y:minibatch_Y})
                epoch_cost += minibatch_cost/num_minibatches
                
            train_costs.append(epoch_cost)
            test_costs.append(comp_cost.eval(feed_dict = {X: X_test, Y: Y_test}))
            train_acc.append(accuracy.eval(feed_dict = {X: X_train, Y: Y_train}))
            test_acc.append(accuracy.eval(feed_dict = {X: X_test, Y: Y_test}))
            #curr_parameters = sess.run(parameters)
            #sample_parameters_list[epoch] = np.dot(curr_parameters['W'+str(sample_hidden_layer)][:, sample_hidden_unit], cache)
            #cache = curr_parameters['W'+str(sample_hidden_layer)][:, sample_hidden_unit]
                                                   
            if print_cost == True and epoch%100 == 0:
                print("Cost after epoch %i: %f" %(epoch, epoch_cost))    
        
        plt.plot(np.squeeze(train_costs))
        plt.plot(np.squeeze(test_costs))
        plt.ylabel('Cost')
        plt.xlabel('Iterations')
        plt.legend(['train_costs', 'test_costs'], loc='upper left')
        plt.show()
        
        plt.plot(np.squeeze(train_acc))
        plt.plot(np.squeeze(test_acc))
        plt.ylabel('Accuracy')
        plt.xlabel('Iterations')
        plt.legend(['train_acc', 'test_acc'], loc='upper left')
        plt.show()
        
        parameters = sess.run(parameters)
        
        print("Train Accuracy: ", accuracy.eval(feed_dict = {X: X_train, Y: Y_train}))
        print("Test Accuracy: ", accuracy.eval(feed_dict = {X: X_test, Y: Y_test}))
        
        return parameters#, sample_parameters_list

In [ ]:
parameters = model(train_data, train_labels_one_hot, test_data, test_labels_one_hot, 
      num_features = train_data.shape[1], 
      num_classes = train_labels_one_hot.shape[1], 
      layers_dims = [train_data.shape[1], 512, 512, train_labels_one_hot.shape[1]], 
      learning_rate = 0.001, momentum = 0.9, num_epochs = 3, minibatch_size = 600, print_cost = True)

In [ ]:
parameters_list

In [ ]:
np.dot(parameters_list[0], np.ones(784, ))

In [ ]:
# All the weights in column 1 belong to first activation unit
# All the weights in first row beling to first feature to all activation units

In [ ]:
for i in range(len(parameters_list)-1):
    print(i)

In [ ]:
np.sum(np.multiply(parameters_list[0], parameters_list[1]), axis = 0)

In [ ]:
parameters_list.keys()

In [ ]:
model = Sequential()
model.add(Dense(512, activation='relu', input_shape=(dimData,)))
model.add(Dense(512, activation='relu'))
model.add(Dense(10, activation='softmax'))

model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_data, train_labels_one_hot, batch_size=256, epochs=5, verbose=1, validation_data=(test_data, test_labels_one_hot))

[test_loss, test_acc] = model.evaluate(test_data, test_labels_one_hot)
print("Evaluation result on Test Data : Loss = {}, accuracy = {}".format(test_loss, test_acc))
image_index = 110
plt.imshow(test_data[image_index].reshape(28, 28),cmap='Greys')
plt.title("Digit from the test data")
plt.show()
pred = model.predict(test_data[image_index].reshape(1,784))
print("Predicted digit:",pred.argmax())
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()